# Part 1 — Python & Data Foundations

**Tasks 1.1–1.6.** This notebook reads the original `data/appointments.csv` twice: first with the `csv` module for type-conversion practice, then independently with pandas for the cleaning pipeline. It generates the required hand-off files for Parts 2 and 3.

Before running, keep the original files unchanged in `data/`: `appointments.csv`, `clinics.csv`, and `followup_notes.csv`.

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import pandas as pd
import requests
import time

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
APPOINTMENTS_FILE = DATA_DIR / 'appointments.csv'
CLINICS_FILE = DATA_DIR / 'clinics.csv'
assert APPOINTMENTS_FILE.exists(), 'Put the original appointments.csv in data/.'
assert CLINICS_FILE.exists(), 'Put the original clinics.csv in data/.'


## Task 1.1 — Read with the `csv` module and reject failed conversions

In [ ]:
with open(APPOINTMENTS_FILE, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)
print('Header:', header)
print('Row count:', len(raw_rows))

def parse_row(row):
    converted = dict(row)
    converted['age'] = int(converted['age'])
    converted['fee_inr'] = int(converted['fee_inr'])
    converted['prior_no_shows'] = int(converted['prior_no_shows'])
    converted['distance_km'] = float(converted['distance_km'])
    return converted

accepted, rejected = [], []
for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)
print('Accepted:', len(accepted), 'Rejected:', len(rejected))
with open(ROOT / 'rejected_rows.json', 'w', encoding='utf-8') as f:
    json.dump(rejected, f, indent=2)


## Task 1.2 — Inspect and clean, pass 1 (restart from original CSV)

In [ ]:
df = pd.read_csv(APPOINTMENTS_FILE)
print('Raw shape:', df.shape)
df.info()
print('Missing values by column:')
print(df.isnull().sum())
print('Exact duplicate rows:', df.duplicated().sum())

df = df.drop_duplicates()
print('Rows after exact duplicate removal:', len(df))
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print('Rows after booking_channel cleaning:', len(df))
print('Booking-channel values:', sorted(df['booking_channel'].dropna().unique()))
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'])
print('Rows after date conversion:', len(df))
df = df.dropna(subset=['appt_date', 'status'])
print('Rows after dropping missing appt_date/status:', len(df))


## Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
invalid_age = (df['age'] < 1) | (df['age'] > 100)
print('Invalid ages:', int(invalid_age.sum()))
df.loc[invalid_age, 'age'] = np.nan
age_median = df['age'].median()
distance_median = df['distance_km'].median()
df['age'] = df['age'].fillna(age_median).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(distance_median)
print('Age median used:', age_median)
print('Distance median used:', distance_median)

cap_counts = {}
for col in ['distance_km', 'fee_inr']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    before = df[col].copy()
    df[col] = np.clip(df[col], lower_fence, upper_fence)
    cap_counts[col] = int((before != df[col]).sum())
    print(f'{col}: Q1={q1}, Q3={q3}, IQR={iqr}, lower={lower_fence}, upper={upper_fence}, capped={cap_counts[col]}')

df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)


## Task 1.4 — Features and NumPy summary

In [ ]:
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)
X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()
column_means = np.mean(X, axis=0)
print('Column means [age, distance_km, fee_inr, lead_days]:', column_means)

long_lead_mask = df['lead_days'] > 14
print('lead_days > 14 count:', int(long_lead_mask.sum()))
print('lead_days > 14 share:', float(long_lead_mask.mean()))

distance = df['distance_km'].to_numpy()
distance_z = (distance - np.mean(distance)) / np.std(distance)
print('Rows with |distance z-score| > 2:', int((np.abs(distance_z) > 2).sum()))
print('No-shows:', int(df['no_show'].sum()), 'of', len(df), '(', round(100 * df['no_show'].mean(), 1), '% )')


## Task 1.5 — Enrich clinics from Open-Meteo

The first hourly temperature is the midnight-UTC stand-in specified in the brief. Each clinic is attempted twice. If both attempts fail, the temperature remains missing and is written as SQL `NULL` later; no temperature is imputed.

In [ ]:
clinics = pd.read_csv(CLINICS_FILE)
temp_values = []
for _, clinic in clinics.iterrows():
    temp_c = None
    failure_reason = None
    params = {
        'latitude': clinic['latitude'],
        'longitude': clinic['longitude'],
        'hourly': 'temperature_2m',
        'forecast_days': 1
    }
    for attempt in range(2):
        try:
            response = requests.get(
                'https://api.open-meteo.com/v1/forecast',
                params=params,
                timeout=10
            )
            if response.status_code != 200:
                failure_reason = f'HTTP status {response.status_code}'
                continue
            payload = response.json()
            hourly_values = payload.get('hourly', {}).get('temperature_2m')
            if not hourly_values:
                failure_reason = 'hourly.temperature_2m missing or empty'
                continue
            temp_c = float(hourly_values[0])
            failure_reason = None
            break
        except requests.exceptions.RequestException as exc:
            failure_reason = f'RequestException: {exc}'
        except (ValueError, TypeError, KeyError) as exc:
            failure_reason = f'Invalid API response: {exc}'
    if temp_c is None:
        print(f"{clinic['clinic_name']}: {failure_reason}; temp_c will be NULL")
    temp_values.append(temp_c)

clinics['temp_c'] = temp_values
clinics.to_csv(ROOT / 'clinics_enriched.csv', index=False)
print('Clinic rows written:', len(clinics))


## Task 1.6 — Export the clean table and write SQL load statements

In [ ]:
columns = [
    'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
    'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type',
    'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show'
]
df = df.sort_values('appointment_id').copy()
for date_col in ['booked_on', 'appt_date']:
    df[date_col] = df[date_col].dt.strftime('%Y-%m-%d')
appointments_clean = df[columns].copy()
print('appointment_id unique:', appointments_clean['appointment_id'].is_unique)
print('Missing values in the 16 export columns:')
print(appointments_clean.isnull().sum())
assert appointments_clean['appointment_id'].is_unique
assert not appointments_clean.isnull().any().any()
appointments_clean.to_csv(ROOT / 'appointments_clean.csv', index=False)

def sql_value(value, quote=False):
    if pd.isna(value):
        return 'NULL'
    if quote:
        return "'" + str(value).replace("'", "''") + "'"
    return str(value)

clinic_text_columns = {'clinic_id', 'clinic_name', 'city', 'tier'}
appointment_text_columns = {
    'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
    'gender', 'booking_channel', 'visit_type', 'reminder_sent', 'status'
}
with open(ROOT / 'load_data.sql', 'w', encoding='utf-8') as sql_file:
    sql_file.write('-- Generated from clinics_enriched.csv and appointments_clean.csv.\n')
    sql_file.write('-- Create tables using 02_queries.sql before running this file.\n\n')
    for _, row in clinics.iterrows():
        values = [sql_value(row[col], col in clinic_text_columns) for col in [
            'clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c'
        ]]
        sql_file.write('INSERT INTO clinics VALUES (' + ','.join(values) + ');\n')
    sql_file.write('\n')
    for _, row in appointments_clean.iterrows():
        values = [sql_value(row[col], col in appointment_text_columns) for col in columns]
        sql_file.write('INSERT INTO appointments VALUES (' + ','.join(values) + ');\n')

print('appointments_clean.csv shape:', appointments_clean.shape)
print('clinics_enriched.csv shape:', clinics.shape)
print('SQL INSERT statements:', len(clinics) + len(appointments_clean))

## Part 1 hand-off and validation notes

The exported `appointments_clean.csv` is the single input for Parts 2 and 3. `clinics_enriched.csv` and `load_data.sql` are the database hand-off. If Open-Meteo cannot be reached after the retry, its `temp_c` fields are intentionally blank and become unquoted `NULL` in SQL.